# MediLink-BioFusion: Multi-Modal Model Training on GPU

This notebook trains the **MediLink-BioFusion** Multi-Modal Gated Transformer model.
The trained weights are saved to a **new file** (`./savepoints/biofusion/model_checkpoint`) without overwriting the original checkpoint.

In [ ]:
# 1. Install & Verify Environment
!pip install -q torch numpy pandas scikit-learn
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Colab Setup: Upload & Extract colab_files.zip (if running in Colab)
import os
if not os.path.exists('src'):
    print('[*] Project files (src/ & data/) not found in Colab.')
    print('[*] Please upload colab_files.zip from your project directory:')
    try:
        from google.colab import files
        uploaded = files.upload()
        !unzip -q colab_files.zip
        print('[OK] Extracted src/ and data/ successfully!')
    except Exception as e:
        print(f'[!] Note: {e}')


In [ ]:
# 3. Load Biological Matrices & Index Data
import json
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

sys.path.append('./src')
from model import build_model
from utils import index_data, convert_tensor, evaluate_model

with open('./data/hyperparameter.json', 'r') as fp:
    hparam = json.load(fp)

# Ensure new save directory is used
hparam['save_path'] = './savepoints/biofusion/'
os.makedirs(hparam['save_path'], exist_ok=True)

print("Loading pre-computed similarity matrices...")
SS_mat = pd.read_pickle('./data/structural_similarity_matrix.pkl')
TS_mat = pd.read_pickle('./data/target_similarity_matrix.pkl')
GS_mat = pd.read_pickle('./data/GO_similarity_matrix.pkl')

mlb, _, idx2label, drugPair2effectIdx = index_data()
x_idx = np.array(list(drugPair2effectIdx.keys()))
y_idx = np.array(list(drugPair2effectIdx.values()))
print(f"Indexed {len(x_idx)} drug pairs across {len(idx2label)} interaction labels.")

In [ ]:
# 4. Prepare Train/Test Split
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(skf.split(x_idx, y_idx))

x_train, y_train = x_idx[train_idx], y_idx[train_idx]
x_test, y_test = x_idx[test_idx], y_idx[test_idx]
print(f"Training samples: {len(x_train)} | Testing samples: {len(x_test)}")

SS_train, TS_train, GS_train, y_train_t = convert_tensor(x_train, y_train, SS_mat, TS_mat, GS_mat, mlb, idx2label)
train_dataset = torch.utils.data.TensorDataset(SS_train, TS_train, GS_train, y_train_t)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=hparam.get('batch_size', 128), shuffle=True)

In [ ]:
# 5. Initialize & Train MediLink-BioFusion Model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(hparam)
model.to(device)

# Train for specified epochs (saves to ./savepoints/biofusion/model_checkpoint)
model.fit(train_loader, repeat=0)

# Save test partition for evaluation
pd.to_pickle([x_test, y_test], './savepoints/biofusion/test_data.pkl')
print("Training complete! Saved new checkpoint to ./savepoints/biofusion/model_checkpoint")

In [ ]:
# 6. Evaluate on Held-Out Test Partition
SS_test, TS_test, GS_test, y_test_t = convert_tensor(x_test, y_test, SS_mat, TS_mat, GS_mat, mlb, idx2label)
model.eval()
with torch.no_grad():
    _, _, _, pred = model(SS_test.to(device), TS_test.to(device), GS_test.to(device))
    pred_bin = (torch.sigmoid(pred) > hparam.get('threshold', 0.5)).int().cpu().numpy()

acc, ma_rc, ma_pc, mi_rc, mi_pc = evaluate_model(y_test_t.int().numpy(), pred_bin)
print("=" * 50)
print("    MediLink-BioFusion Evaluation Results     ")
print("=" * 50)
print(f"Accuracy:        {acc:.4f} ({acc*100:.2f}%)")
print(f"Micro Recall:    {mi_rc:.4f} ({mi_rc*100:.2f}%)")
print(f"Micro Precision: {mi_pc:.4f} ({mi_pc*100:.2f}%)")
print(f"Macro Recall:    {ma_rc:.4f} ({ma_rc*100:.2f}%)")
print(f"Macro Precision: {ma_pc:.4f} ({ma_pc*100:.2f}%)")
print("=" * 50)